# MobileNetV3

This notebook trains a MobileNetV3 Classification model using MultiTask learning.

1. Install dependancies (Cell 1)
2. Mount Google Drive & set paths (Cell 2)
3. Load split IDs text files (Cell 3)
4. Train (Cell 4-11)
10. Save best (Cell 12)



# Set paths

In [ ]:
from pathlib import Path

EXCEL = Path.cwd().parent.parent/"dataset/BTXRD/dataset.xlsx"
IMAGES = Path.cwd().parent.parent/"dataset/BTXRD/images"
ANNOTATIONS = Path.cwd().parent.parent/"dataset/BTXRD/Annotations"

# Load split IDs




In [ ]:
def read_ids(p):
    with open(p) as f:
        return [ln.strip() for ln in f if ln.strip()]

train_ids = read_ids(Path.cwd().parent.parent/"evaluation/train_ids.txt")
validation_ids = read_ids(Path.cwd().parent.parent/"evaluation/val_ids.txt")
test_ids = read_ids(Path.cwd().parent.parent/"evaluation/test_ids.txt")

print(f"Training Set: {len(train_ids)}, Validation Set: {len(validation_ids)}, Test Set: {len(test_ids)}")

# Hyperparameters




In [ ]:
import torch

BATCH_HEAD = 8
BATCH_FULL = 4
EPOCHS_HEAD = 15
EPOCHS_FULL = 25
LEARNING_RATE_HEAD = 5e-4
LEARNING_RATE_FULL = 1e-4
WEIGHT_DECAY = 2e-4
SEGMENTATION_WEIGHTING = 0.5
IMAGE_HEAD_WEIGHTING = 0.3
CLASSIFICATION_THRESHOLD = 0.30
NAME_MODEL = "mobilenetv3"
NAME_BACKBONE = "mobilenetv3_large_100"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device used for training:", DEVICE)

# preprocessing




In [ ]:
import numpy
import pandas
import cv2
import json
import re
import os

df = pandas.read_excel(EXCEL, engine="openpyxl")

# Paths for each image
df["image_path"] = df["image_id"].map(lambda x: os.path.join(IMAGES, x))
df["annot_path"] = df["image_id"].map(lambda x: os.path.join(ANNOTATIONS, x.replace(".jpeg", ".json")))

# Context columns
df["age"] = pandas.to_numeric(df.get("age"))

def gender(x):
    s = str(x).strip().lower()
    if s in ["F"]:
        return 1.0
    if s in ["M"]:
        return 0.0
    return 0.0

df["gender_f"] = df.get("gender", "").map(gender)

train_df = df[df["image_id"].isin(train_ids)].reset_index(drop=True)
val_df   = df[df["image_id"].isin(validation_ids)].reset_index(drop=True)
test_df  = df[df["image_id"].isin(test_ids)].reset_index(drop=True)

age_mean = 35.57109479305741
age_std  = 20.997566907248853

# Augmentations & dataset class




In [ ]:
import albumentations
import torch
from torch.utils.data import Dataset
import numpy
import cv2, json, os

# Augmentations
aug_train = albumentations.Compose([albumentations.HorizontalFlip(p=0.5),
                                    albumentations.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.7),
                                    albumentations.ElasticTransform(p=0.3, alpha=10, sigma=120 * 0.05),
                                    albumentations.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.6),
                                    albumentations.GaussianBlur(blur_limit=(3, 7), p=0.4),
                                    albumentations.GridDropout(ratio=0.05, random_offset=True, p=0.5),
                                    albumentations.Resize(height=512, width=512)])

aug_val = albumentations.Compose([albumentations.LongestMaxSize(max_size=512),
                                  albumentations.PadIfNeeded(min_height=512, min_width=512, border_mode=cv2.BORDER_CONSTANT)])

def normalise_image(image):
    image = image.astype(numpy.float32)/255.0
    mean = numpy.array([0.485,0.456,0.406], dtype=numpy.float32)
    std = numpy.array([0.229,0.224,0.225], dtype=numpy.float32)
    return (image - mean)/std

def context(row, stats):
    age = row.get("age")
    age_z = (age - stats["age_mean"])/(stats["age_std"])
    frontal = float(row.get("frontal",0))
    lateral = float(row.get("lateral",0))
    oblique = float(row.get("oblique",0))
    return numpy.array([age_z, float(row.get("gender_f", 0.0)), frontal, lateral, oblique], dtype=numpy.float32)

class BTXRDDataset(Dataset):
    def __init__(self, frame, aug, context):
        self.f = frame.reset_index(drop=True)
        self.aug = aug
        self.context = context
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self):
        return len(self.f)

    def __getitem__(self, idx):
        row = self.f.iloc[idx]
        classification_label = torch.tensor(row.tumor, dtype=torch.float32)
        image = cv2.imread(row.image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        #CLAHE
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l,a,b = cv2.split(lab)
        l = self.clahe.apply(l)
        image = cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB)

        # Build mask
        height, width = image.shape[:2]
        mask = numpy.zeros((height, width), dtype=numpy.float32)
        if int(row.tumor) == 1 and os.path.isfile(row.annot_path):
              with open(row.annot_path, "r") as f:
                  annotation = json.load(f)
              for shape in annotation.get("shapes", []):
                  points = numpy.array(shape.get("points", []), dtype=numpy.int32)
                  if points.ndim == 2 and points.shape[0] >= 3:
                      cv2.fillPoly(mask, [points], 1)

              pass

        output = self.aug(image=image, mask=mask)
        image, mask = output["image"], output["mask"]

        x = torch.from_numpy(normalise_image(image).transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None].astype(numpy.float32))
        ctx = torch.from_numpy(context(row, self.context)).float()

        return x, classification_label, mask, ctx


# Dataloaders

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

#Weighted training
counts = train_df.tumor.value_counts().sort_index()
sample_weights = train_df.tumor.map(lambda l: 1.0 / counts[int(l)]).astype(float).values
sampler = WeightedRandomSampler(weights=torch.tensor(sample_weights, dtype=torch.float32), num_samples=len(sample_weights), replacement=True)

context_stats = {"age_mean": age_mean, "age_std": age_std}
training_dl_head = DataLoader(BTXRDDataset(train_df, aug_train, context_stats), batch_size=BATCH_HEAD, sampler=sampler, shuffle=False, num_workers=0, pin_memory=True, persistent_workers=False)
validation_dl_head = DataLoader(BTXRDDataset(val_df, aug_val, context_stats), batch_size=BATCH_HEAD, shuffle=False, num_workers=0, pin_memory=True, persistent_workers=False)
training_dl_full = DataLoader(BTXRDDataset(train_df, aug_train, context_stats), batch_size=BATCH_FULL, sampler=sampler, shuffle=False, num_workers=0, pin_memory=True, persistent_workers=False)
validation_dl_full = DataLoader(BTXRDDataset(val_df, aug_val, context_stats), batch_size=BATCH_FULL, shuffle=False, num_workers=0, pin_memory=True, persistent_workers=False)

# Model definition

In [ ]:
import timm
from torch import nn
from torch.nn import functional
import torch

class MobileNetV3(nn.Module):
    def __init__(self, model_name='mobilenetv3_large_100', pretrained=True, ctx_dim=5):
        super().__init__()
        self.encoder = timm.create_model('mobilenetv3_large_100', pretrained=pretrained, features_only=True)
        ec = self.encoder.feature_info.channels()
        self.decoder_c4 = nn.Conv2d(ec[-1], 64, 1)
        self.decoder_c3 = nn.Conv2d(ec[-2], 48, 1)
        self.decoder_c2 = nn.Conv2d(ec[-3], 24, 1)
        self.decoder_d3 = nn.Conv2d(64 + 48, 64, 3, padding=1)
        self.decoder_d2 = nn.Conv2d(64 + 24, 32, 3, padding=1)
        self.decoder_d1 = nn.Conv2d(32, 16, 3, padding=1)
        self.decoder_d0 = nn.Conv2d(16, 8, 3, padding=1)

        #Head
        self.segmentation_head = nn.Conv2d(8, 1, 1)
        self.pool= nn.AdaptiveAvgPool2d(1)
        self.classification_head = nn.Linear(ec[-1], 1)
        self.context_mlp= nn.Sequential(nn.Linear(5, 32), nn.ReLU(inplace=True), nn.BatchNorm1d(32), nn.Dropout(0.10), nn.Linear(32, 16), nn.ReLU(inplace=True))
        self.fusion_head = nn.Linear(ec[-1] + 16, 1)

    def forward(self, x, context=None):
        feats = self.encoder(x)
        c4,c3,c2 = feats[-1], feats[-2], feats[-3]

        pooled  = self.pool(c4).flatten(1)
        classification_image = self.classification_head(pooled)
        context_emb   = self.context_mlp(context)
        fused = torch.cat([pooled, context_emb], dim=1)
        classificaiton_fused = self.fusion_head(fused)

        # Decoder
        p4 = self.decoder_c4(c4); p3 = self.decoder_c3(c3); p2 = self.decoder_c2(c2)
        d3 = functional.interpolate(p4, size=p3.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.decoder_d3(torch.cat([d3, p3], dim=1))
        d2 = functional.interpolate(d3, size=p2.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.decoder_d2(torch.cat([d2, p2], dim=1))
        d1 = self.decoder_d1(functional.interpolate(d2, scale_factor=2, mode='bilinear', align_corners=False))
        d0 = self.decoder_d0(functional.interpolate(d1, scale_factor=2, mode='bilinear', align_corners=False))
        y_seg = self.segmentation_head(functional.interpolate(d0, size=x.shape[2:], mode='bilinear', align_corners=False))

        return classificaiton_fused, y_seg, classification_image

# Training utilities

In [ ]:
import numpy
from torch import nn
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from torch import amp
from tqdm.auto import tqdm
import torch

criterion_classification = nn.BCEWithLogitsLoss()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        iflat, tflat = probs.contiguous().view(-1), targets.contiguous().view(-1)
        inter = (iflat * tflat).sum()
        return 1 - ((2*inter + self.smooth)/(iflat.sum() + tflat.sum() + self.smooth))

def dice_metric(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    iflat, tflat = probs.contiguous().view(-1), targets.contiguous().view(-1)
    inter = (iflat * tflat).sum()
    return (2*inter + smooth)/(iflat.sum() + tflat.sum() + smooth)

def epoch_metrics(yt, yp):
    yt, yp = numpy.array(yt), numpy.array(yp)
    return roc_auc_score(yt, yp), accuracy_score(yt, (yp > CLASSIFICATION_THRESHOLD).astype(int)), log_loss(yt, yp, labels=[0,1])

def make_opt(model, lr):
    return torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=WEIGHT_DECAY)

def make_sched(opt, ep, steps):
    return torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=opt.param_groups[0]['lr'], steps_per_epoch=steps, epochs=ep,
        pct_start=0.15, div_factor=25.0, final_div_factor=1e4
    )

def run_epoch(dl, train, model, opt=None, sched=None):
    model.train(train)
    scaler = amp.GradScaler(enabled=(DEVICE == "cuda"))
    yh_fused, yh_img, yt = [], [], []
    losses, dice_scores = [], []

    for x, classification_label , segmentation_label, ctx in tqdm(dl, leave=False):
        x = x.to(DEVICE); classification_label  = classification_label .to(DEVICE).unsqueeze(1); segmentation_label = segmentation_label.to(DEVICE)
        ctx = ctx.to(DEVICE)

        with (torch.no_grad() if not train else torch.enable_grad()):
            with amp.autocast(device_type=("cuda" if DEVICE=="cuda" else "cpu"), enabled=(DEVICE=="cuda")):
                classification_fused, segmentation_output, classification_image = model(x, ctx)
                loss_classification_fused = criterion_classification (classification_fused, classification_label)
                loss_classification_image  = criterion_classification(classification_image, classification_label)
                loss_segmentation = DiceLoss()(segmentation_output, segmentation_label)
                total_loss = loss_classification_fused + SEGMENTATION_WEIGHTING * loss_segmentation + IMAGE_HEAD_WEIGHTING * loss_classification_image

        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(total_loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            prev_scale = scaler.get_scale()
            scaler.step(opt)
            scaler.update()
            if sched and scaler.get_scale() >= prev_scale:
                sched.step()

        yh_fused.extend(torch.sigmoid(classification_fused).detach().cpu().numpy().ravel())
        yh_img.extend(torch.sigmoid(classification_image).detach().cpu().ravel())
        yt.extend(classification_label.detach().cpu().ravel())
        losses.append(total_loss.item())
        dice_scores.append(dice_metric(segmentation_output, segmentation_label).item())

    auc_fused, accuracy_fused, _ = epoch_metrics(yt, yh_fused)
    auc_image, accuracy_image, _ = epoch_metrics(yt, yh_img)
    return (auc_fused, accuracy_fused, auc_image, accuracy_image, float(numpy.mean(losses)), float(numpy.mean(dice_scores)))

# Train Head

In [ ]:
model = MobileNetV3(model_name=NAME_BACKBONE , pretrained=True, ctx_dim=5).to(DEVICE)
best_auc, best_state = 0.0, None

# Freeze encoder
for p in model.encoder.parameters():
    p.requires_grad = False

opt  = make_opt(model, LEARNING_RATE_HEAD)
sched = make_sched(opt, EPOCHS_HEAD, len(training_dl_head))

for ep in range(1, EPOCHS_HEAD + 1):
    training = run_epoch(training_dl_head, True,  model, opt, sched)
    validation = run_epoch(validation_dl_head, False, model)
    (train_auc_full, train_acc_full, train_auc, train_accuracy, training_loss, training_dice) = training
    (validation_auc_full, validation_accuracy_full, validation_auc, validation_accuracy, validation_loss, validation_dice) = validation
    print(f"{ep:02d}: "
          f"Training: (AUC {train_auc_full:.3f}, Dice {training_dice:.3f}) "
          f"Validation: (AUC {validation_accuracy_full:.3f}, Dice {validation_dice:.3f})")
    if validation_auc_full > best_auc:
        best_auc, best_state = validation_auc_full, {k: v.cpu() for k, v in model.state_dict().items()}

# Train Full

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

# Unfreeze all
for p in model.parameters():
    p.requires_grad = True

# Discriminative learning rates
FULL_LR  = LEARNING_RATE_FULL * 0.25
HEAD_LR = LEARNING_RATE_FULL

enc_params   = [p for n, p in model.named_parameters() if n.startswith("encoder")]
other_params = [p for n, p in model.named_parameters() if not n.startswith("encoder")]

opt = torch.optim.AdamW([{"params": enc_params,"lr": FULL_LR},{"params": other_params,"lr": HEAD_LR}], weight_decay=WEIGHT_DECAY)

sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[FULL_LR, HEAD_LR], steps_per_epoch=len(training_dl_head), epochs=EPOCHS_FULL,
    pct_start=0.30, div_factor=10.0, final_div_factor=1000.0)

for ep in range(1, EPOCHS_FULL + 1):
    tr = run_epoch(training_dl_head, True,  model, opt, sched)
    vl = run_epoch(validation_dl_head, False, model)
    (tr_auc_f, tr_acc_f, tr_auc_i, tr_acc_i, tr_loss, tr_dice) = tr
    (vl_auc_f, vl_acc_f, vl_auc_i, vl_acc_i, vl_loss, vl_dice) = vl
    print(f"{ep:02d}: "f"Training: (AUC {tr_auc_f:.3f}, Dice {tr_dice:.3f}) "f"Validation: (AUC {vl_auc_f:.3f}, Dice {vl_dice:.3f})")
    if vl_auc_f > best_auc:
        best_auc, best_state = vl_auc_f, {k: v.cpu() for k, v in model.state_dict().items()}

# Save Model

In [ ]:
model.load_state_dict(best_state)
model_path = os.path.join(Path.cwd().parent.parent / "models", f"{NAME_MODEL}_best.pt")
torch.save(best_state, model_path)
print(f"Saved best model: {model_path}")